# Exercise 12

This exercise is based on Chapter 10 (Clustering & Regionalization) of the Geographic Data Science book.

The material can be found in: `GSP538/gds_book/notebooks/10_clustering_and_regionalization.ipynb`

#### Answer the following written questions

There is a blank Markdown cell after each question for your answer (double click in the blank cell to type your answer). Be sure to run your Markdown cells to format your answers.

1. Considering the vocabulary of this chapter, what is the difference between a cluster, a geodemographic cluster and a region?

A **cluster** is a group of observations that are more similar to one another (in attribute space) than to members of other groups. Cluster labels are arbitrary integers with no inherent ordering. Clusters can be spatially fragmented—members may be scattered across a map with no geographic continuity.

A **geodemographic cluster** is a cluster applied to spatially referenced demographic or socioeconomic data. The multivariate clustering compresses variation across many dimensions into a single categorical variable that can be visualized on a map. Each cluster is described by its *profile*—a summary of attribute means that characterizes what areas in that cluster are like.

A **region** has the same attribute-coherence property as a cluster, but adds a geographic constraint: all members must be spatially contiguous. You can travel from any member of a region to any other member without leaving the region. Regions are produced by **regionalization**, which embeds clustering logic but enforces connectivity through a spatial weights matrix.

2. Why do we generally need to standardize data before applying a clustering or regionalization method?

Clustering algorithms rely on computing distances between observations in attribute space. If variables are measured on different scales, variables with larger magnitudes will dominate the distance calculation, effectively drowning out variables with smaller magnitudes. For example, a variable measured in hundreds of thousands (like median house value) would overwhelm a variable measured between 0 and 1 (like a Gini coefficient). Standardizing the data puts all variables on a comparable scale, ensuring that each variable contributes proportionally to the distance calculation and thus to the clustering result.

3. When the authors talk about "distance" in this chapter, it does not relate to distance on the ground. What does distance refer to here? Explain. (Hint: this is somewhat related to the previous question.)

In this chapter, "distance" refers to **statistical distance in attribute (feature) space**, not geographic distance on the ground. When two observations are said to be "close," it means their values across all clustering variables are similar—for example, their Euclidean distance in the multi-dimensional space defined by the standardized variables is small. This is related to the previous question because standardization is needed precisely to ensure that the distance calculation treats all variables equally, rather than being dominated by whichever variable has the largest numerical range.

4. How is `robust_scale` different from `scale`? When is `robust_scale` advantageous over `scale`? Why?

`scale` standardizes data by subtracting the mean and dividing by the standard deviation: $z = \frac{x_i - \bar{x}}{\sigma}$. It centers the data at zero with unit variance. However, both the mean and standard deviation are sensitive to outliers.

`robust_scale` instead uses the **median** and the **interquartile range (IQR)**: $z = \frac{x_i - \tilde{x}}{Q_{75} - Q_{25}}$. Because the median and IQR are resistant to extreme values, `robust_scale` is advantageous when the data contains outliers or is heavily skewed. In such cases, `scale` would produce misleading standardized values (the mean and standard deviation would be pulled by extreme values), whereas `robust_scale` gives a more representative transformation of the typical spread of the data.

5. Go to the following web page. Near the top, click the "Randomly" button. Then click "Uniform Points." Add four centroids by clicking the "Add Centroid" button bellow the points. Then click "Go!". Keep clicking the "Update Centroids" and "Reassign Points" buttons until the centroids stop moving. You can then reset the tool and run it again with the same or different settings if you choose.
https://www.naftaliharris.com/blog/visualizing-k-means-clustering/

   In your own words, explain how k-means works. Do NOT explain the tool above; explain generally how k-means starts and works its way to a solution.

K-means begins by selecting a number of clusters, *k*, that the analyst specifies in advance. The algorithm then proceeds iteratively:

1. **Initialization**: All observations are randomly assigned to one of the *k* clusters.
2. **Update centroids**: The multivariate mean (centroid) of each cluster is computed across all input variables.
3. **Reassign observations**: Each observation is compared to every cluster centroid and reassigned to the cluster whose centroid it is closest to in attribute space.
4. **Repeat**: Steps 2 and 3 are repeated. Each iteration, some observations switch clusters, and the centroids shift accordingly.
5. **Convergence**: The algorithm stops when no observations change cluster membership between iterations.

At convergence, every observation is closer to the centroid of its own cluster than to the centroid of any other cluster.

6. Why do you need to specify the number of clusters in advance for agglomerative hierarchical clustering (AHC)? 

AHC builds a complete hierarchy (dendrogram) that starts with every observation as its own singleton cluster and progressively merges the two most similar clusters at each step, all the way down to a single cluster containing everything. This hierarchy contains a valid clustering for *every* value of *k* from 1 to *n*. The analyst must specify the number of clusters to tell the algorithm where to "cut" the hierarchy—i.e., at which level of aggregation to stop merging. Without specifying *k*, the algorithm would either stop at singletons (useless) or merge everything into one cluster (also useless).

7. What is the role of the spatial weights matrix in a regionalization method? What kind of spatial weights matrix is needed?

In regionalization, the spatial weights matrix serves as a **connectivity constraint**. It is passed to the clustering algorithm (e.g., as the `connectivity` parameter in `AgglomerativeClustering`) and restricts which observations can be merged: two observations can only be placed in the same cluster if they are connected (neighbors) in the weights graph. This forces the resulting clusters to be spatially contiguous regions.

A **contiguity-based** spatial weights matrix is needed—typically **queen contiguity**, where two spatial units are neighbors if they share any boundary point (edge or vertex). This ensures that members of each resulting region are geographically connected, so you can travel from any member to any other member within the same region without crossing into a different region.

8. What is the isoperimetric quotient? Conceptually, how is it computed? What is its range?

The **isoperimetric quotient (IPQ)** is a measure of geographic compactness that compares a region's shape to a circle. It is computed as:

$$IPQ = \frac{4\pi A}{P^2}$$

where $A$ is the area of the region and $P$ is its perimeter. Conceptually, it compares the actual area of the region to the area of a circle with the same perimeter.

The IPQ ranges from **0 to 1**. A value near **1** indicates a compact shape approaching a circle, while a value near **0** indicates an elongated, irregular, or fragmented shape. Regionalized solutions tend to have higher IPQ values than unconstrained clustering solutions because their spatially contiguous regions share internal boundaries and have more compact shapes.

9. Why do we expect that agglomerative hierarchical clustering (AHC) without a spatial constraint will have a better "goodness of fit" score than AHC with the spatial constraint?

This is an inherent tradeoff: regionalization is **constrained** clustering. The spatial constraint limits which observations the algorithm is allowed to merge, reducing its freedom to optimize cluster coherence in attribute space. Mathematically, a constrained optimization problem can never achieve a better objective score than the equivalent unconstrained problem—if the unconstrained solution happened to also satisfy the spatial constraint, they would tie, but that is extremely unlikely. In practice, the improved geographic coherence (compact, contiguous regions) comes at the cost of reduced statistical goodness of fit (e.g., lower Calinski-Harabasz scores), because some observations are forced into clusters based on spatial proximity rather than pure attribute similarity.

#### The following questions require you to run Python code.

The cell below imports the GeoPandas package and the standard extra stuff to make things run smoother. Run this cell. 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import geopandas as gpd

10. You will be using invasive species data for this exercise.

    > The 2018 Invasive Plant Treatment Prioritization (IPTP) analysis of the Forest and Woodland Health Program (FH) at the Arizona Department of Forestry and Fire Management (DFFM). The purpose of the analysis was to assess the Arizona landscape and identify invasive plant treatment priority areas based on management criteria.

    - Read in `central_az_invasive.geojson` from the `data` folder
    - Use the `explore()` method to visualize the spatial units and extent of the data
    - Use the `info()` method to get some basic information on the data
    - Give a brief description of what you are working with

In [ ]:
gdf = gpd.read_file('data/central_az_invasive.geojson')
gdf.explore()

In [ ]:
gdf.info()

The dataset contains 14,260 hexagonal spatial units covering central Arizona, with 28 columns. The data comes from the 2018 Invasive Plant Treatment Prioritization (IPTP) analysis. Each hexagon has attributes including fire risk, riparian area, protected species counts, wildlife corridor length, threat scores, treated acreage, wildland-urban interface (WUI) area, and impervious surface area—both as raw data values and as normalized index values. The geometry column contains MultiPolygon geometries in EPSG:4326 (WGS 84).

11. The full report accompanying the data can be found at: https://dffm.az.gov/sites/default/files/media/2018_AZ_IPTP_Report_DFFM.pdf. The table on page 13 lists the eight inputs to their prioritization model and how they were computed. 

    The cell below contains two lists:
    - `index_cols`: corresponds to the first column in the page 13 table from the PDF (same order as table)
    - `data_cols`: corresponds to the second column in the page 13 table from the PDF (same order as table)

    <br>

    Do the following:
    - Run `describe()` on the `data_cols` columns (output is one table)
    - Run `describe()` on the `index_cols` columns (output is one table)
    - Based on those two tables, which set of eight columns can be used directly as inputs for clustering? Explain.

In [ ]:
data_cols = ['az_fri_rc2','fl_ri_we_acre','HDMS_spp_count','corridor_miles',
             'threat_score','treated_acre','SILVIS_WUI_ac','nlcd_imperv']

index_cols = ['fire_risk_index','riparian_index','protected_spp_index','corridor_index',
              'threat_index','treated_index','wui_index','undeveloped_index']

In [ ]:
gdf[data_cols].describe()

In [ ]:
gdf[index_cols].describe()

The **`index_cols`** can be used directly as inputs for clustering without standardization. The `describe()` output shows that all eight index columns have values roughly on the same scale (approximately 0 to 1), meaning no single variable will dominate the distance calculation.

In contrast, the **`data_cols`** have wildly different scales: for example, `fl_ri_we_acre` ranges up to ~640, `HDMS_spp_count` is a small integer (0–14), and `nlcd_imperv` ranges up to ~78. If used directly, the high-magnitude variables like riparian acres and WUI acres would overwhelm the clustering, and variables with smaller ranges would be effectively ignored. These columns would need standardization before clustering.

12. Investigate the distributions of the inputs and relationships between them.
    - Run Seaborn `pairplot` on the `data_col` columns using `kind='reg'` and `diag_kind='kde'` (Note: it might be a little slow to run)
    - Pick at least two inputs and discuss their univariate distributions
    - Pick at least two pairs of inputs and discuss their relationships to one another
    - Note: in the discussions use common terms from the first column of the PDF table, not the variable names

In [ ]:
import seaborn as sns
sns.pairplot(gdf[data_cols], kind='reg', diag_kind='kde')

**Univariate distributions:**
- **Riparian area (fl_ri_we_acre):** Heavily right-skewed—most hexagons have very little riparian area, but a few have very high values (up to ~640 acres). The KDE shows a sharp peak near zero with a long tail.
- **Threat score (threat_score):** Extremely right-skewed with the vast majority of hexagons having a score of 0. Only a small fraction of hexagons have any recorded invasive species threats.

**Bivariate relationships:**
- **WUI area (SILVIS_WUI_ac) vs. impervious surface (nlcd_imperv):** These show a moderate positive relationship—hexagons with more wildland-urban interface tend to have more impervious surface, which makes sense since both reflect development and urbanization.
- **Fire risk (az_fri_rc2) vs. riparian area (fl_ri_we_acre):** There is a weak positive association. Hexagons with higher fire risk tend to have somewhat more riparian area, possibly reflecting vegetated areas that are both ecologically important and fire-prone.

13. k-means (baseline)
    - Run k-means clustering using the `index_col` columns and set k to 5 (Note: do not scale the data)
    - Create a map showing the clusters
    - Describe the spatial pattern of the results
    - Note: add `np.random.seed(111)` to the beginning of your code cell so that you get the same result as the instructor

In [ ]:
from sklearn.cluster import KMeans

np.random.seed(111)
km_base = KMeans(n_clusters=5)
km_base.fit(gdf[index_cols])
gdf['km_base'] = km_base.labels_

gdf.plot(column='km_base', categorical=True, legend=True, figsize=(10, 8))

The k-means baseline map shows five clusters that are spatially fragmented—hexagons of the same cluster are scattered across the study area rather than forming contiguous regions. One cluster (the largest with ~8,900 hexagons) dominates the landscape and appears to cover broad swaths of lower-priority land. Other clusters form patches and speckles throughout the map, reflecting areas with distinct combinations of index values. The spatial pattern reveals that the clustering is driven purely by attribute similarity with no geographic constraint, resulting in salt-and-pepper mixing of cluster labels across the map.

14. k-means (source data)
    - Run k-means clustering using the `data_col` columns and set k to 5 (Note: do not scale the data)
    - Create a map showing the clusters using `.plot()`
    - Are the results generally similar to the k-means (baseline) map or generally different? Explain.
    - Note: when you store the labels, use a different column name from previous solutions
    - Note: add `np.random.seed(222)` to the beginning of your code cell so that you get the same result as the instructor    

In [ ]:
np.random.seed(222)
km_data = KMeans(n_clusters=5)
km_data.fit(gdf[data_cols])
gdf['km_data'] = km_data.labels_

gdf.plot(column='km_data', categorical=True, legend=True, figsize=(10, 8))

The results are **generally different** from the baseline map. Using the raw source data columns (which have very different scales) causes variables with larger magnitudes—particularly riparian area (`fl_ri_we_acre`, up to ~640) and WUI area (`SILVIS_WUI_ac`, up to ~640)—to dominate the distance calculation, while variables with smaller ranges are effectively ignored. This produces a clustering driven primarily by those two high-magnitude variables. One cluster overwhelmingly dominates (~11,000 hexagons), and the remaining clusters isolate relatively small groups of hexagons with extreme values in the high-magnitude variables. This demonstrates why standardization (or using pre-normalized index values) is important for balanced clustering.

15. k-means (k=10)
    - Run k-means clustering using the `index_col` columns and set k to 10 (Note: do not scale the data)
    - Create a map showing the clusters using `.plot()`
    - Describe some differences and similarities between this map and the k-means (baseline) map.
    - Note: when you store the labels, use a different column name from previous solutions
    - Note: add `np.random.seed(333)` to the beginning of your code cell so that you get the same result as the instructor    

In [ ]:
np.random.seed(333)
km_10 = KMeans(n_clusters=10)
km_10.fit(gdf[index_cols])
gdf['km_10'] = km_10.labels_

gdf.plot(column='km_10', categorical=True, legend=True, figsize=(10, 8))

**Similarities:** Like the baseline (k=5), the k=10 map is spatially fragmented with clusters scattered across the study area. One large cluster still dominates (~6,800 hexagons), similar to the dominant cluster in the baseline. The overall geographic extent and coverage look similar.

**Differences:** With 10 clusters instead of 5, the map is more complex and shows finer distinctions between areas. The previously large clusters are subdivided into smaller, more specialized groups. Several clusters are quite small (100–500 hexagons), picking up niche combinations of index values that were lumped together in the k=5 solution. The map appears more "noisy" with more color variation, making spatial patterns harder to visually interpret.

16. k-means (baseline - statistical analysis)

    Note: This question uses the "baseline" k-means results you first created (i.e., using `index_cols` and k=5)
    - How many hexagons are in each cluster?
    - Compute the mean value for the `data_cols` columns for each cluster (Note: this should follow the style used in the book that transposes the rows and columns)
    - Describe each cluster based on the statistical data 
      - Hint: for each cluster identify which inputs are extreme (high or low) for that cluster
      - Note: use real words for the inputs from the PDF, not the variable names

In [ ]:
# How many hexagons in each cluster
gdf.groupby('km_base').size()

In [ ]:
# Mean values per cluster (transposed for readability)
gdf.groupby('km_base')[data_cols].mean().T

**Cluster 0** (758 hexagons): Characterized by **very high riparian area** (~350 acres, far above all other clusters). Moderate fire risk and protected species count. This cluster represents riparian-rich areas.

**Cluster 1** (8,918 hexagons): The largest cluster, representing the "background" landscape. It has **low values across nearly all inputs**—low fire risk, low riparian area, few protected species, low threat scores, near-zero treated acreage, and low WUI area. These are generally low-priority hexagons.

**Cluster 2** (1,015 hexagons): Distinguished by **very high WUI area** (~453 acres) and **high impervious surface** (~14%), along with **high corridor miles** (~12). This cluster represents urbanized or peri-urban areas with significant wildland-urban interface.

**Cluster 3** (2,602 hexagons): Has the **highest fire risk** (~5.2) among all clusters, combined with relatively low values for other inputs. Low impervious surface and low threat scores. These are fire-prone, relatively undeveloped areas.

**Cluster 4** (967 hexagons): Characterized by the **highest threat score** (~1.6, far above other clusters) and relatively **high protected species counts** (~0.7). Moderate corridor miles and impervious surface. This cluster represents areas with the most active invasive species threats.

17. Agglomerative hierarchical clustering (AHC)

    - Run AHC clustering using the `index_col` columns and set the number of clusters to 5 (Note: do not scale the data; this runs slower than k-means)
    - Create a map showing the clusters using `.plot()`
    - Describe the spatial pattern of the results relative to k-means (baseline)
    - Note: when you store the labels, use a different column name from previous solutions

In [ ]:
from sklearn.cluster import AgglomerativeClustering

ahc = AgglomerativeClustering(linkage='ward', n_clusters=5)
ahc.fit(gdf[index_cols])
gdf['ahc5'] = ahc.labels_

gdf.plot(column='ahc5', categorical=True, legend=True, figsize=(10, 8))

Like the k-means baseline, the AHC results are spatially fragmented—clusters are scattered across the map without spatial contiguity. However, the cluster size distribution differs: AHC produces two larger clusters (~7,800 and ~3,900 hexagons) and three smaller clusters (~600–1,000 each), compared to k-means which had one dominant cluster (~8,900) with the remainder more evenly distributed. The overall spatial pattern is broadly similar to k-means in terms of fragmentation, but the specific assignment of hexagons to clusters varies because AHC uses a hierarchical merging approach (Ward linkage) rather than iterative centroid reassignment.

18. Regionalization

    - Run AHC clustering using the `index_col` columns, set the number of clusters to 5 and use a queen weights matrix (Note: do not scale the data)
    - Create a map showing the clusters using `.plot()`
    - Describe the spatial pattern of the results relative to k-means (baseline)
    - Note: when you store the labels, use a different column name from previous solutions

In [ ]:
from libpysal.weights import Queen

w = Queen.from_dataframe(gdf)
ahc_reg = AgglomerativeClustering(linkage='ward', connectivity=w.sparse, n_clusters=5)
ahc_reg.fit(gdf[index_cols])
gdf['reg5'] = ahc_reg.labels_

gdf.plot(column='reg5', categorical=True, legend=True, figsize=(10, 8))

The regionalization map is strikingly different from the k-means baseline. Instead of spatially fragmented clusters scattered across the map, the regionalization produces **spatially contiguous regions**—large, connected blocks of hexagons that form coherent geographic areas. Each region spans a continuous portion of the study area. This is because the queen contiguity weights matrix constrains the AHC algorithm to only merge neighboring hexagons, ensuring that every member of a region is geographically connected to every other member. The result is much easier to interpret as distinct geographic zones, though the trade-off is reduced statistical coherence within each region compared to the unconstrained solutions.

19. Compare the k-means (baseline) result to the regionalization result

    - Feature coherence
      - Compute the Calinski Harabasz score using `index_cols` for k-means (baseline)
      - Compute the Calinski Harabasz score using `index_cols` for regionalization
    - Solution similarity
      - Compute the Adjusted Mutual Information Score between k-means (baseline) and regionalization
    - Interpret the results
    - Note: the textbook uses a bunch of fancy code to compute multiple values using for loops; you just need three one-liners (two for feature coherence and one for solution similarity)

In [ ]:
from sklearn import metrics

# Calinski-Harabasz scores
print("CH score k-means (baseline):", metrics.calinski_harabasz_score(gdf[index_cols], gdf['km_base']))
print("CH score regionalization:", metrics.calinski_harabasz_score(gdf[index_cols], gdf['reg5']))

# Adjusted Mutual Information
print("AMI (k-means baseline vs regionalization):", metrics.adjusted_mutual_info_score(gdf['km_base'], gdf['reg5']))

**Feature coherence (Calinski-Harabasz):** The k-means baseline has a much higher CH score (~4,837) compared to the regionalization (~1,465). Higher CH scores indicate better-defined clusters in attribute space, so the unconstrained k-means solution achieves significantly better statistical fit. This is expected because the spatial constraint in regionalization limits the algorithm's freedom to optimize cluster coherence—it must sacrifice attribute-space fit to achieve geographic contiguity.

**Solution similarity (Adjusted Mutual Information):** The AMI score is approximately 0.15, which is low. This indicates that the two solutions assign hexagons to clusters in **very different ways**—there is little agreement between the k-means and regionalization labels. This makes sense because k-means groups hexagons purely by attribute similarity (regardless of location), while regionalization is heavily constrained by spatial adjacency, producing fundamentally different groupings.